In [42]:
import pandas as pd

In [43]:
# 불러오기 및 내역 확인
df1=pd.read_csv('data/movie/tmdb_5000_credits.csv')
df2=pd.read_csv('data/movie/tmdb_5000_movies.csv')
df1.shape, df2.shape


((4803, 4), (4803, 20))

In [44]:
#df1,df2의 열내역이 동일한지 확인(flood문 출력)
df1['title'].equals(df2['title'])

True

In [45]:
# 열제목 확인
df1.columns, df2.columns

(Index(['movie_id', 'title', 'cast', 'crew'], dtype='object'),
 Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
        'original_title', 'overview', 'popularity', 'production_companies',
        'production_countries', 'release_date', 'revenue', 'runtime',
        'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
        'vote_count'],
       dtype='object'))

In [46]:
#df1의 movie_id와 df2의 id 가 동일한지 확인(T/F)
df1['movie_id'].equals(df2['id'])

True

In [47]:
#df1의 movie_id를 df2의 id로 확인
df1.rename(columns={'movie_id':'id'}, inplace=True)
df1.head(2) #변경여부 확인용


,id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [48]:
#df1에서 필요한 부분만 추출
df_temp=df1[['id','cast','crew']]

In [49]:
#id를 기준으로 df1,df2를 결합(merge 사용) 기준은 df2
df=df2.merge(df_temp,on='id')
df.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'cast', 'crew'],
      dtype='object')

In [50]:
#출력된 부분에서 vote_count//vote_average 확인
df[['vote_count','vote_average']].head()

,vote_count,vote_average
0,11800,7.2
1,4500,6.9
2,4466,6.3
3,9106,7.6
4,2124,6.1


In [51]:
#평점 가중치 구하는 공식
#WR=(v/v(v+m)*r)+(m/(v+m)*c)
# c, m 지정(평균, 상위 90%의 댓글 개수)
c=df['vote_average'].mean()
c

m=df['vote_count'].quantile(0.9)
m

np.float64(1838.4000000000015)

In [52]:
#df_movie를 df에서 복사하여 상위 90 이하의 댓글수를 제외한 내림차순 분류
df_movies=df.copy()
filt=df['vote_count']>=m
df_movies=df_movies[filt] # df_movie에 filt 함수 적용하기
df.shape, df_movies.shape #결과물 확인
df_movies['vote_count'].sort_values() #vote_count 기준으로 내림차순 정렬


2585     1840
195      1851
2454     1859
597      1862
1405     1864
        ...  
788     10995
16      11776
0       11800
65      12002
96      13752
Name: vote_count, Length: 481, dtype: int64

In [53]:
#가중치 구하는 공식을 함수로 만들기
def wetighted_rating(x):
    m=df['vote_count'].quantile(0.9)
    c=df['vote_average'].mean()
    v=x['vote_count']
    r=x['vote_average']
    return (v/(v+m)*r)+(m/(v+m)*c)



In [ ]:
#df_movies에 score 열을 만드는데, 점수를 1개씩 넣ㅇ어(axis=1) 가중치를 구한 것을 넣음 
df_movies['score']=df_movies.apply(wetighted_rating,axis=1)
df_movies[['title','score']]

#df_movies를 score로 내림차순으로 구함(출력은 'title','vote_count','vote_average','score'에서 상위 5개만 )
df_movies=df_movies.sort_values('score',ascending=False)
df_movies[['title','vote_count','vote_average','score']].head(5)

,title,vote_count,vote_average,score
1881,The Shawshank Redemption,8205,8.5,8.059258
662,Fight Club,9413,8.3,7.939256
65,The Dark Knight,12002,8.2,7.920020
3232,Pulp Fiction,8428,8.3,7.904645
96,Inception,13752,8.1,7.863239
